In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm

import gpytorch

from Functions import EDFA_Modelling, Fibre_ModelLoad
import CustomKernels.LaplacianKernel as LK
import CustomKernels.LaplacianKernelUncertainty as LKUncertain
import CustomKernels.UncertainKernel as UncertainKernel
import CustomKernels.UncertainMeanConstant as UncertainMean
import CustomKernels.UncertainMeanZero as UncertainZeroMean

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

uncertain = False

In [2]:
inputPowers = pd.read_csv("data/EDFA 6/Inputs.csv", index_col=0)
outputPowers = pd.read_csv("data/EDFA 6/Outputs.csv", index_col=0)
perms = pd.read_csv("data/EDFA 6/perms.csv", index_col=0)

EDFA_settings = pd.read_csv("data/EDFA 6/EDFA_setting.csv", index_col=0)
EDFA1_setting = EDFA_settings["0"]


onChannels = perms.copy()

onChannels[onChannels > -1] = 1
onChannels[onChannels == -1] = 0

scaledInputs, OutputPowersArr, initialVariance = EDFA_Modelling.preprocess_data(inputPowers, outputPowers, EDFA1_setting, onChannels, uncertain)

### EDFA1 Model

In [3]:
EDFAID = "1"
edfa1_outputs, edfa1_variances = EDFA_Modelling.make_predictions(scaledInputs, initialVariance, uncertain, EDFAID, device)

d:\ch982\OneDrive - University of Cambridge\Documents\JLT 2025\5Spans\Functions\EDFA_Modelling.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(p

### Fibre 1 Model

In [4]:
fibre1_output, fibre1_variance = Fibre_ModelLoad.fibrePredictions("fibre 1", onChannels, edfa1_outputs, edfa1_variances)

### EDFA2 Model

In [5]:
fibre1_output_df = pd.DataFrame(fibre1_output)
combinedInputs = pd.concat([fibre1_output_df, EDFA_settings[["0", "1"]].iloc[:len(fibre1_output_df)]], axis=1)

fibre1_variance = pd.DataFrame(fibre1_variance)
fibre1_variance["16"] = 0
fibre1_variance["17"] = 0
fibre1_variance = fibre1_variance.to_numpy()

scalerIn = MinMaxScaler()
scaledInputs = scalerIn.fit_transform(combinedInputs.to_numpy())

EDFAID = "2"
edfa2_outputs, edfa2_variances = EDFA_Modelling.make_predictions(scaledInputs, fibre1_variance, uncertain, EDFAID, device)

d:\ch982\OneDrive - University of Cambridge\Documents\JLT 2025\5Spans\Functions\EDFA_Modelling.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(p

### Fibre 2 Model

In [6]:
fibre2_output, fibre2_variance = Fibre_ModelLoad.fibrePredictions("fibre 2", onChannels, edfa2_outputs, edfa2_variances)

### EDFA3 Model

In [7]:
fibre2_output_df = pd.DataFrame(fibre2_output)
combinedInputs = pd.concat([fibre2_output_df, EDFA_settings[["0", "1", "2"]].iloc[:len(fibre2_output_df)]], axis=1)

fibre2_variance = pd.DataFrame(fibre2_variance)
fibre2_variance["16"] = 0
fibre2_variance["17"] = 0
fibre2_variance["18"] = 0
fibre2_variance = fibre2_variance.to_numpy()

scalerIn = MinMaxScaler()
scaledInputs = scalerIn.fit_transform(combinedInputs.to_numpy())

EDFAID = "3"
edfa3_outputs, edfa3_variances = EDFA_Modelling.make_predictions(scaledInputs, fibre2_variance, uncertain, EDFAID, device)

d:\ch982\OneDrive - University of Cambridge\Documents\JLT 2025\5Spans\Functions\EDFA_Modelling.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(p

### Fibre 3 Model

In [8]:
fibre3_output, fibre3_variance = Fibre_ModelLoad.fibrePredictions("fibre 3", onChannels, edfa3_outputs, edfa3_variances)

### EDFA4 Model

In [9]:
fibre3_output_df = pd.DataFrame(fibre3_output)
combinedInputs = pd.concat([fibre3_output_df, EDFA_settings[["0", "1", "2", "3"]]], axis=1)

fibre3_variance = pd.DataFrame(fibre3_variance)
fibre3_variance["16"] = 0
fibre3_variance["17"] = 0
fibre3_variance["18"] = 0
fibre3_variance["19"] = 0
fibre3_variance = fibre3_variance.to_numpy()

scalerIn = MinMaxScaler()
scaledInputs = scalerIn.fit_transform(combinedInputs.to_numpy())

EDFAID = "4"
edfa4_outputs, edfa4_variances = EDFA_Modelling.make_predictions(scaledInputs, fibre3_variance, uncertain, EDFAID, device)

d:\ch982\OneDrive - University of Cambridge\Documents\JLT 2025\5Spans\Functions\EDFA_Modelling.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(p

### Fibre 4 Model

In [10]:
fibre4_output, fibre4_variance = Fibre_ModelLoad.fibrePredictions("fibre 4", onChannels, edfa4_outputs, edfa4_variances)

### EDFA5 Model

In [11]:
fibre4_output = pd.DataFrame(fibre4_output)
combinedInputs = pd.concat([fibre4_output, EDFA_settings[["0", "1", "2", "3", "4"]]], axis=1)

fibre4_variance = pd.DataFrame(fibre4_variance)
fibre4_variance["16"] = 0
fibre4_variance["17"] = 0
fibre4_variance["18"] = 0
fibre4_variance["19"] = 0
fibre4_variance["20"] = 0
fibre4_variance = fibre4_variance.to_numpy()

scalerIn = MinMaxScaler()
scaledInputs = scalerIn.fit_transform(combinedInputs.to_numpy())

EDFAID = "5"
edfa5_outputs, edfa5_variances = EDFA_Modelling.make_predictions(scaledInputs, fibre4_variance, uncertain, EDFAID, device)

d:\ch982\OneDrive - University of Cambridge\Documents\JLT 2025\5Spans\Functions\EDFA_Modelling.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(p

c:\Users\ch982\.conda\envs\EDFA_env\Lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


### Fibre 5 Model

In [12]:
fibre5_output, fibre5_variance = Fibre_ModelLoad.fibrePredictions("fibre 5", onChannels, edfa5_outputs, edfa5_variances)

### EDFA 6 Model

In [13]:
fibre5_output = pd.DataFrame(fibre5_output)
combinedInputs = pd.concat([fibre5_output, EDFA_settings.iloc[:len(fibre5_output)]], axis=1)

fibre5_variance = pd.DataFrame(fibre5_variance)
fibre5_variance["16"] = 0
fibre5_variance["17"] = 0
fibre5_variance["18"] = 0
fibre5_variance["19"] = 0
fibre5_variance["20"] = 0
fibre5_variance["21"] = 0
fibre5_variance = fibre5_variance.to_numpy()

scalerIn = MinMaxScaler()
scaledInputs = scalerIn.fit_transform(combinedInputs.to_numpy())

### Split Data for EDFA 6 Model

In [14]:
numTraining = 300
testIndex = 300

X_train, y_train, X_test, y_test = EDFA_Modelling.split_data(scaledInputs, OutputPowersArr, numTraining, testIndex, uncertain, fibre5_variance)

### Train EDFA 6 Model

In [15]:
model, likelihood, training_data = EDFA_Modelling.train_model(X_train, y_train, device, uncertain)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [03:13<00:00,  5.16it/s]


### Evaluate EDFA 6 Model

In [16]:
l1, l2, preds, vars = EDFA_Modelling.test_model(model, likelihood, X_test, y_test, numTraining, uncertain)

In [17]:
l1, l2

(0.19232734496929685, 0.08016670912032858)

In [18]:
EDFA_Modelling.save_model(model, training_data, uncertain, "6")